In [8]:
from tensorflow.keras import layers,models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [9]:

img_height,img_width = 128,128
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 30,
    zoom_range = 0.3,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    horizontal_flip = True,
    brightness_range=[0.7,1.3],
    validation_split = 0.2,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)
base_model.trainable = False

In [10]:

train_datagen = train_datagen.flow_from_directory(
    'Dataset-2/train',
    target_size=(img_height,img_width),
    class_mode = "binary",
    batch_size = batch_size  
)

test_datagen = test_datagen.flow_from_directory(
    'Dataset-2/test',
    target_size=(img_height,img_width),
    class_mode = "binary",
    batch_size = batch_size  
)

val_datagen = val_datagen.flow_from_directory(
    'Dataset-2/val',
    target_size=(img_height,img_width),
    class_mode = "binary",
    batch_size = batch_size  
)



Found 11627 images belonging to 2 classes.
Found 375 images belonging to 2 classes.
Found 499 images belonging to 2 classes.


In [11]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
    
)

In [12]:
model.fit(train_datagen,validation_data = val_datagen,epochs = 10)

Epoch 1/10
364/364 [==============================] - 129s 348ms/step - loss: 0.5290 - accuracy: 0.7283 - val_loss: 0.4326 - val_accuracy: 0.8397
Epoch 2/10
364/364 [==============================] - 117s 320ms/step - loss: 0.4697 - accuracy: 0.7667 - val_loss: 0.4256 - val_accuracy: 0.8257
Epoch 3/10
364/364 [==============================] - 116s 319ms/step - loss: 0.4510 - accuracy: 0.7780 - val_loss: 0.3888 - val_accuracy: 0.8437
Epoch 4/10
364/364 [==============================] - 115s 315ms/step - loss: 0.4307 - accuracy: 0.7887 - val_loss: 0.4509 - val_accuracy: 0.7836
Epoch 5/10
364/364 [==============================] - 115s 316ms/step - loss: 0.4251 - accuracy: 0.7969 - val_loss: 0.4496 - val_accuracy: 0.7836
Epoch 6/10
364/364 [==============================] - 116s 318ms/step - loss: 0.4158 - accuracy: 0.7959 - val_loss: 0.4508 - val_accuracy: 0.7816
Epoch 7/10
364/364 [==============================] - 115s 315ms/step - loss: 0.4059 - accuracy: 0.8064 - val_loss: 0.4361 -

In [13]:
test_loss, test_acc = model.evaluate(test_datagen)

print("Test Accuracy:", test_acc)
print("Test Loss:", test_loss)

12/12 [==============================] - 5s 405ms/step - loss: 0.4668 - accuracy: 0.7653
Test Accuracy: 0.765333354473114
Test Loss: 0.46678751707077026


In [14]:
print(train_datagen.class_indices)
print(test_datagen.class_indices)
print(val_datagen.class_indices)

{'disaster': 0, 'non_disaster': 1}
{'disaster': 0, 'non_disaster': 1}
{'disaster': 0, 'non_disaster': 1}


In [15]:
model.save("Disaster_detector.keras")

In [2]:
import tensorflow as tf
import numpy as np
import cv2
model = tf.keras.models.load_model("Disaster_detector.keras")
IMG_SIZE = 128

def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)
    prediction = model.predict(img)[0][0]
    
    if prediction > 0.5:
        print("Non Disaster")
    else:
        print("Disaster")

predict_image(r"C:\Users\User\Downloads\download.png")

1/1 [==============================] - 0s 456ms/step
Non Disaster


In [6]:
from PIL import Image
import os

def clean_dataset(folder):
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                path = os.path.join(root, file)
                try:
                    img = Image.open(path)
                    img.verify()
                except:
                    print("Removing:", path)
                    os.remove(path)

clean_dataset(r"D:\Project Disaster management\Dataset-2\train")

Removing: D:\Project Disaster management\Dataset-2\train\non_disaster\02_0069.png
